# Project DAT305 - Motion Regognition from IMU data

By Andreas Nesse & Simen Kolbjørnsen

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.functions import *
from src.classes import *

import keras
import tensorflow

## TODO:

### Administrative
1. Deside on technology:

* Keras & Tensorflow
* Numpy & Pandas


2. Set problem/task boundaries and structure


3. Report


4. Presentation


5. Review

### Implementation
1. Import data and structure data files
2. Load data into notebook
3. Clean dataset
4. Structure the data for the model
5. Implement and structure the model
6. Train the model on the data
7. Feedback loop

### Implementation method:
1 step is ~500 + seconds long on avg
This means, with a $\Delta t$ of 0.0125 s or 12.5 ms between the IMU readings, 1 step is $500 / 12.5 = 40$ points. In each IMU reading we have 6 values, 3 for acceleration and 3 for gyroscope data.

By entering the data raw to the model, this means that the model has to take in 240 input values.

To compress this we can use the magnitude of the acceleration and gyroscope data, meaning that instead of 6 values per point, we get 2. Again leading to 80 input values for the model

Furthermore it could be possible to just use statistical/mathematical properties of the data. Like computing the mean, sd, variance, max, min etc. This again could make the input size less than 10.

We want to make the model as compressed as possible. So converting all values to 8-bit integers instead of the default 64-bit would reduce the size by 1/4. Also using a more traditional approach, like using regression models instead of a NN will also be optimal as long as it manages to give the right results.

We have 2 627 548 data points from 75 different recording sessions from 31 different people. Meaning we have som variety in the data so that the model doesn't get alligned only to one type of person (e.g. tall or short people). So say each input to the model is 40 points, that leaves us with $2 627 548 / 40 = 65 688.7$ inputs of data to be used to train/validate/test. So just over $2^{16}$.


## 1. DATA
### 1.1 Loading the data

In [ ]:

try:
    print(f"Data already loaded: {len(data_original)} files")
except NameError:
    BASE_URL = "https://raw.githubusercontent.com/simpleLoc/SIMUL-dataset/refs/heads/master/data/by-person/"
    
    file_URLs = pd.read_csv("https://raw.githubusercontent.com/andynesse/DAT305/refs/heads/main/file_urls.csv", header=None)[0]

    data_original = load_data(BASE_URL, file_URLs[:20])
    print(len(data_original))

data = data_original.copy()

Save collected data to own repo:

In [ ]:
# data_copy = data.copy()
# for i, df in enumerate(data_copy):
#     df.to_csv(f"data/raw/data_{i}")

Assigning ids to the dataframes, then combining them into one big df. Then saves it into a 2.6 million row long csv file ... 2.27 GB file...

In [ ]:
# data_copy = data.copy()

# data_copy_marked = [df.assign(id=i) for i, df in enumerate(data_copy)]


In [ ]:
# big_data = pd.concat(data_copy_marked)
# big_data.describe()

In [ ]:
# big_data.to_csv("data/combined_data.csv")

Select and rename relevant parameters and do describe on all the dataframes:

In [ ]:
columns = {'Timestamp' : 'ts', ' Activity' : 'activity', ' AccelX_3' : 'aX', ' AccelY_3' : 'aY', ' AccelZ_3' : 'aZ', ' GyroX_3' : 'gX', ' GyroY_3' : 'gY', ' GyroZ_3' : 'gZ'}
for i, df in enumerate(data):
    df = df.rename(columns=columns)
    df = df[list(columns.values())]
    print(df.describe())
    data[i] = df


Check if we have null values (Fortunately we have none):

In [ ]:
for df in data:
    print(df.isnull().sum())

Here we plot the magnetude of the acc and gyro for a dataframe:

In [ ]:
data_sample = data[0]
#data_sample = data_sample[data_sample['activity'] != -1]

def magnitude(x, y, z):
    return np.sqrt(x**2 + y**2 + z**2)

ts = data_sample['ts'].to_numpy()
aX = data_sample['aX'].to_numpy()
aY = data_sample['aY'].to_numpy()
aZ = data_sample['aZ'].to_numpy()
gX = data_sample['gX'].to_numpy()
gY = data_sample['gY'].to_numpy()
gZ = data_sample['gZ'].to_numpy()

plt.plot(ts, magnitude(aX, aY, aZ), label="Acceleration")
plt.plot(ts, magnitude(gX, gY, gZ), label="Gyro")

#plt.xlim(left = 155, right = 160)
plt.legend()
plt.show()



Piechart of how many of the datapoints are walking or still in the df:

In [ ]:
activities = data_sample['activity']
still = activities[activities == 0]
pie = [len(still)/len(activities), 1 - len(still)/len(activities)]
plt.pie(pie, labels=["still", "in motion"],colors=("black","pink"))
plt.show()

A closer look on the magnetude of acc and gyro, so that we can see the repeating signal of walking motion:

In [ ]:
data_sample = data[0]

ts = data_sample['ts'].to_numpy()
aX = data_sample['aX'].to_numpy()
aY = data_sample['aY'].to_numpy()
aZ = data_sample['aZ'].to_numpy()
gX = data_sample['gX'].to_numpy()
gY = data_sample['gY'].to_numpy()
gZ = data_sample['gZ'].to_numpy()

plt.plot(ts, magnitude(aX, aY, aZ), label="Acceleration")
plt.plot(ts, magnitude(gX, gY, gZ), label="Gyro")
dt = 3
tx = 105
plt.xlim(left = tx, right = tx+dt)
plt.legend()
plt.show()

The amount of datapoints we have, combined from all the files:

In [ ]:
sum(len(d) for d in data)

Here we made a window class for combining datapoints into windows for training:

In [ ]:
class Window:
    SIZE = 40
    activities = {0 : 0, 1 : 1, 2 : 1, 3 : 1, 4 : 0, 5 : 0}
    def __init__(self, df, activity):
        self.data = df
        self.activity = Window.activities[int(activity)]

    def to_windows(df):
        result = []
        left = 0
        while left < len(df) - Window.SIZE:
            window = df.iloc[left : left + Window.SIZE]
            left += Window.SIZE
            if window['activity'].std() != 0.0:
                continue
            result.append(Window(window, window.iloc[0, 1]))
        return result



Here we plot the activity as like a digital signal, and also the acc x for the windows we created in a dataframe:

In [ ]:
windows_all_df = dfs_to_windows(data, window_by_parameters, 'activity', ['aX', 'aY', 'aZ', 'gX', 'gY', 'gZ'])
windows_all_df.describe()

In [ ]:
windows_magnitudes_df = dfs_to_windows(data, window_by_magnitude, 'activity', [("acc", 'aX', 'aY', 'aZ'), ("gyro", 'gX', 'gY', 'gZ')])
windows_magnitudes_df.describe()

In [ ]:
windows_values_df = dfs_to_windows(data, window_by_values, 'activity', [("acc", 'aX', 'aY', 'aZ'), ("gyro", 'gX', 'gY', 'gZ')])
windows_values_df.describe()

## Regression

In [ ]:
all_points_data = TrainingData(windows_all_df, 'activity')
magnitude_data = TrainingData(windows_magnitudes_df, 'activity')
statistical_data = TrainingData(windows_values_df, 'activity')

#### Make regression model:

In [ ]:
all_points_model = regression_model(all_points_data.n_cols)
magnitude_model = regression_model(magnitude_data.n_cols)
statistical_model = regression_model(statistical_data.n_cols)

Using the fit method to train and test the model at the same time. validation split set to 0.3 means 30% of the data set to validation, while the rest of 70% is used for training. Here we use 10 epochs:

In [ ]:
all_points_history = all_points_data.fit(all_points_model)

In [ ]:
magnitude_history = magnitude_data.fit(magnitude_model)

In [ ]:
statistical_history = statistical_data.fit(statistical_model)

We then plot the results in a graph:

In [ ]:
pd.DataFrame(all_points_history.history).plot(figsize=(8, 5), grid=True)
pd.DataFrame(magnitude_history.history).plot(figsize=(8, 5), grid=True)
pd.DataFrame(statistical_history.history).plot(figsize=(8, 5), grid=True)

#plt.grid(True)
#plt.gca().set_ylim(0, 1) # set the vertical range to [0-1]
plt.show()

In [ ]:
pd.DataFrame(all_points_history.history)

In [ ]:
pd.DataFrame(magnitude_history.history)

In [ ]:
pd.DataFrame(statistical_history.history)